# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*
Signal 1: CTR vs. position (behind FlyRank's CTR-fix logic) — does CTR
actually get worse as position gets worse?

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"CREATE SECRET hf_secret (TYPE huggingface, TOKEN '{hf_token}');")

BASE = "hf://datasets/FlyRank/internship-warehouse"
FACT_MARCH = f"{BASE}/fact_content_daily_performance/month=2026-03/data_0.parquet"

signal1 = con.execute(f"""
    SELECT
      CASE WHEN gsc_avg_position <= 3 THEN '1-3'
           WHEN gsc_avg_position <= 10 THEN '4-10'
           WHEN gsc_avg_position <= 20 THEN '11-20'
           ELSE '21+' END AS position_tier,
      COUNT(*) AS n,
      AVG(gsc_clicks * 1.0 / gsc_impressions) AS mean_ctr
    FROM read_parquet('{FACT_MARCH}')
    WHERE gsc_data_available IS TRUE AND gsc_impressions > 0
    GROUP BY position_tier
    ORDER BY position_tier
""").df()
print(signal1)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  position_tier        n  mean_ctr
0           1-3   727362  0.004756
1         11-20   519223  0.002770
2           21+   908354  0.001289
3          4-10  1456122  0.003473


Signal 2: volume behind "quick-win" logic — do pages in the page-2 zone
(position 11-20) with real impressions form a meaningful pool?

In [2]:
signal2 = con.execute(f"""
    SELECT
      CASE WHEN gsc_avg_position BETWEEN 11 AND 20 THEN 'quick_win_zone (11-20)'
           ELSE 'other' END AS zone,
      COUNT(*) AS n,
      AVG(gsc_impressions) AS mean_impressions,
      SUM(CASE WHEN gsc_impressions >= 100 THEN 1 ELSE 0 END) AS n_with_100plus_impressions
    FROM read_parquet('{FACT_MARCH}')
    WHERE gsc_data_available IS TRUE
    GROUP BY zone
""").df()
print(signal2)

                     zone        n  mean_impressions  \
0                   other  3158813         80.850698   
1  quick_win_zone (11-20)   452248         55.866146   

   n_with_100plus_impressions  
0                    577897.0  
1                     60711.0  


**My rule (plain words):** prioritize pages in the "quick-win zone" (position
11-20) that already get real impressions — close to page 1, visible enough
to matter. Secondary flag: pages ranking well (top 10) but with unusually
low CTR for their position — a possible on-page/snippet issue.

**Reason codes:** `quick_win_zone_high_impressions`, `good_position_low_ctr`, `monitor_only`
**Action label:** `review_for_refresh` or `no_action`

**Verdicts:**
- Signal 1 (CTR vs. position): **CONFIRMED** — CTR declines monotonically as
  position worsens (0.476% → 0.347% → 0.277% → 0.129%), matching the CTR-fix
  logic's assumption.
- Signal 2 (quick-win zone volume): **MIXED** — the zone has a real, usable
  pool (60,711 rows with 100+ impressions), but it is not unusually rich
  compared to other positions (13.4% vs 18.3% clear the 100-impression bar,
  and mean impressions are lower: 55.9 vs 80.9). The "quick win" framing
  needs to lean on position proximity to page 1, not on volume being special
  here.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

Building the score: aggregate each page's March performance (avg position,
total impressions, CTR), then score = quick-win zone proximity + visibility.
One reason code per page, one action label. Written to
work/outputs/baseline_action_score.csv.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import pandas as pd
# Aggregate to one row per page over March (avoids row-level noise / duplicates)
page_level = con.execute(f"""
    SELECT
      content_hash_id,
      AVG(gsc_avg_position) AS avg_position,
      SUM(gsc_impressions) AS total_impressions,
      SUM(gsc_clicks) AS total_clicks
    FROM read_parquet('{FACT_MARCH}')
    WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id
""").df()

page_level["ctr"] = page_level.apply(
    lambda r: r["total_clicks"] / r["total_impressions"] if r["total_impressions"] > 0 else 0,
    axis=1
)

def score_row(row):
    in_quick_win_zone = 11 <= row["avg_position"] <= 20
    has_real_traffic = row["total_impressions"] >= 100
    good_position = row["avg_position"] <= 10
    low_ctr_for_position = row["ctr"] < 0.01

    if in_quick_win_zone and has_real_traffic:
        # Band 300-400: closer to position 11 = more winnable = higher score
        score = 400 - row["avg_position"]
        reason = "quick_win_zone_high_impressions"
        action = "review_for_refresh"
    elif good_position and low_ctr_for_position and has_real_traffic:
        # Band 200-300: scaled by how much traffic is being wasted
        score = 200 + min(row["total_impressions"] / 1000, 99)
        reason = "good_position_low_ctr"
        action = "review_for_refresh"
    else:
        # Band 0-100: never outranks a real action row
        score = min(row["total_impressions"] / 10000, 99)
        reason = "monitor_only"
        action = "no_action"

    return pd.Series([score, reason, action])

page_level[["score", "reason_code", "action"]] = page_level.apply(score_row, axis=1)

ranked_queue = page_level.sort_values("score", ascending=False).reset_index(drop=True)

print(f"Total pages scored: {len(ranked_queue)}")
print(ranked_queue["reason_code"].value_counts())
print(ranked_queue["action"].value_counts())

os.makedirs("work/outputs", exist_ok=True)
ranked_queue.to_csv("work/outputs/baseline_action_score.csv", index=False)
print("Saved to work/outputs/baseline_action_score.csv")

ranked_queue.head(10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Total pages scored: 176738
reason_code
monitor_only                       105988
good_position_low_ctr               52474
quick_win_zone_high_impressions     18276
Name: count, dtype: int64
action
no_action             105988
review_for_refresh     70750
Name: count, dtype: int64
Saved to work/outputs/baseline_action_score.csv


,content_hash_id,avg_position,total_impressions,total_clicks,ctr,score,reason_code,action
0,content_ecadb0c274a899a5,11.000066,972.0,0.0,0.000000,388.999934,quick_win_zone_high_impressions,review_for_refresh
1,content_7ad8245311afbf47,11.000823,405.0,0.0,0.000000,388.999177,quick_win_zone_high_impressions,review_for_refresh
2,content_89f5ab311968e3f1,11.000847,507.0,0.0,0.000000,388.999153,quick_win_zone_high_impressions,review_for_refresh
3,content_7d35753744841366,11.001459,1610.0,0.0,0.000000,388.998541,quick_win_zone_high_impressions,review_for_refresh
4,content_517249bed12ac982,11.001729,357.0,2.0,0.005602,388.998271,quick_win_zone_high_impressions,review_for_refresh
5,content_d62c65eadf55b0e1,11.002970,468.0,0.0,0.000000,388.997030,quick_win_zone_high_impressions,review_for_refresh
6,content_c21f948068a52b54,11.004413,201.0,6.0,0.029851,388.995587,quick_win_zone_high_impressions,review_for_refresh
7,content_3474f47f0f323858,11.004428,5268.0,124.0,0.023538,388.995572,quick_win_zone_high_impressions,review_for_refresh
8,content_9691e083d932e7a5,11.004471,1321.0,8.0,0.006056,388.995529,quick_win_zone_high_impressions,review_for_refresh
9,content_ad089b42732c04a9,11.005217,209.0,0.0,0.000000,388.994783,quick_win_zone_high_impressions,review_for_refresh


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Reviewing the top 20 by score: for each, the action, reason code, a
confidence note, and what would make it wrong.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top20 = ranked_queue.head(20).copy()

def confidence_note(row):
    if row["total_impressions"] >= 1000:
        return "high confidence — solid traffic volume backs this position"
    elif row["total_impressions"] >= 300:
        return "medium confidence — modest traffic, position estimate less stable"
    else:
        return "low confidence — thin traffic, position could be noisy"

top20["confidence"] = top20.apply(confidence_note, axis=1)

for i, row in top20.iterrows():
    print(f"{i+1}. {row['content_hash_id']}")
    print(f"   Action: {row['action']} | Reason: {row['reason_code']}")
    print(f"   Confidence: {row['confidence']}")
    print(f"   Why here: position {row['avg_position']:.2f} (edge of page 2), "
          f"{int(row['total_impressions'])} impressions, ctr {row['ctr']:.4f}")
    print(f"   Would be wrong if: low-impression rows are too thin to trust the "
          f"position estimate, or zero-click rows reflect a snippet/title issue "
          f"rather than a rankable position problem.\n")

1. content_ecadb0c274a899a5
   Action: review_for_refresh | Reason: quick_win_zone_high_impressions
   Confidence: medium confidence — modest traffic, position estimate less stable
   Why here: position 11.00 (edge of page 2), 972 impressions, ctr 0.0000
   Would be wrong if: low-impression rows are too thin to trust the position estimate, or zero-click rows reflect a snippet/title issue rather than a rankable position problem.

2. content_7ad8245311afbf47
   Action: review_for_refresh | Reason: quick_win_zone_high_impressions
   Confidence: medium confidence — modest traffic, position estimate less stable
   Why here: position 11.00 (edge of page 2), 405 impressions, ctr 0.0000
   Would be wrong if: low-impression rows are too thin to trust the position estimate, or zero-click rows reflect a snippet/title issue rather than a rankable position problem.

3. content_89f5ab311968e3f1
   Action: review_for_refresh | Reason: quick_win_zone_high_impressions
   Confidence: medium confidence —

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak picks: the entire top 20 sits at position 11.00-11.01 — because the
score formula (400 - avg_position) rewards proximity to exactly position 11,
it fills the queue with the closest-to-11 pages before ever considering
position 12+. This means the top 20 isn't diverse across the quick-win zone,
it's a tight cluster at one edge of it — a real limitation, since two pages
at position 11.0 and 19.9 are treated as wildly different priority even
though both are legitimately "quick win" candidates.

Additionally, several top rows have low impressions (140-500) and zero
clicks — the rule doesn't yet weigh traffic magnitude within the position
band, so a 200-impression page and a 5,000-impression page

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
inputs_used = ["avg_position", "total_impressions", "total_clicks", "ctr"]
excluded_as_leaky = ["future report_date rows", "any product-generated flag/score"]
print("Inputs used (all observable at decision time):", inputs_used)
print("Deliberately excluded (leakage risk):", excluded_as_leaky)
print(f"Position range in top 20: {top20['avg_position'].min():.2f} to {top20['avg_position'].max():.2f}")

Inputs used (all observable at decision time): ['avg_position', 'total_impressions', 'total_clicks', 'ctr']
Deliberately excluded (leakage risk): ['future report_date rows', 'any product-generated flag/score']
Position range in top 20: 11.00 to 11.01


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.